In [19]:
from pathlib import Path
import random
import re
from collections import Counter

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

In [20]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)

Device: cuda


In [21]:
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

PROCESSED_DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

train_df = pd.read_csv(
    PROCESSED_DATA_DIR / "train_77.csv"
)

validation_df = pd.read_csv(
    PROCESSED_DATA_DIR / "validation_77.csv"
)

test_df = pd.read_csv(
    PROCESSED_DATA_DIR / "test_77.csv"
)

print("Train:", train_df.shape)
print("Validation:", validation_df.shape)
print("Test:", test_df.shape)

Train: (9149, 2)
Validation: (1961, 2)
Test: (1961, 2)


In [22]:
def tokenize(text: str):

    if not isinstance(text, str):
        raise TypeError("text必须是字符串")

    text = text.lower().strip()

    tokens = re.findall(
        r"[a-z0-9]+(?:'[a-z]+)?",
        text
    )

    return tokens

In [23]:
token_counter = Counter()

for text in train_df["text"]:
    token_counter.update(
        tokenize(text)
    )


MIN_FREQUENCY = 2

PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"


word_to_id = {
    PAD_TOKEN: 0,
    UNK_TOKEN: 1
}


for token, count in token_counter.items():

    if count >= MIN_FREQUENCY:
        word_to_id[token] = len(word_to_id)


id_to_word = {
    index: word
    for word, index
    in word_to_id.items()
}


VOCAB_SIZE = len(word_to_id)

PAD_ID = word_to_id[PAD_TOKEN]
UNK_ID = word_to_id[UNK_TOKEN]

print("Vocabulary size:", VOCAB_SIZE)

Vocabulary size: 1459


In [24]:
MAX_LENGTH = 50


def encode_text(
    text,
    vocabulary
):

    tokens = tokenize(text)

    token_ids = [
        vocabulary.get(
            token,
            vocabulary[UNK_TOKEN]
        )
        for token in tokens
    ]

    return token_ids

In [25]:
def encode_pad_and_get_length(
    text,
    vocabulary,
    max_length
):

    token_ids = encode_text(
        text,
        vocabulary
    )

    # 真正送给LSTM的有效长度
    sequence_length = min(
        len(token_ids),
        max_length
    )

    # 截断
    token_ids = token_ids[
        :max_length
    ]

    # Padding
    padding_length = (
        max_length
        - len(token_ids)
    )

    token_ids = (
        token_ids
        + [PAD_ID] * padding_length
    )

    return (
        token_ids,
        sequence_length
    )

In [26]:
class_names = sorted(
    train_df["category"].unique()
)

label_to_id = {
    label: index
    for index, label
    in enumerate(class_names)
}

id_to_label = {
    index: label
    for label, index
    in label_to_id.items()
}

NUM_CLASSES = len(class_names)

print("Number of classes:", NUM_CLASSES)

Number of classes: 77


In [27]:
class Banking77BiLSTMDataset(Dataset):

    def __init__(
        self,
        dataframe,
        vocabulary,
        label_mapping,
        max_length
    ):

        self.dataframe = (
            dataframe
            .reset_index(drop=True)
            .copy()
        )

        self.vocabulary = vocabulary
        self.label_mapping = label_mapping
        self.max_length = max_length


    def __len__(self):

        return len(
            self.dataframe
        )


    def __getitem__(
        self,
        index
    ):

        row = self.dataframe.iloc[
            index
        ]

        text = row["text"]
        category = row["category"]

        (
            input_ids,
            sequence_length
        ) = encode_pad_and_get_length(
            text=text,
            vocabulary=self.vocabulary,
            max_length=self.max_length
        )

        label_id = self.label_mapping[
            category
        ]

        return {
            "input_ids": torch.tensor(
                input_ids,
                dtype=torch.long
            ),

            "length": torch.tensor(
                sequence_length,
                dtype=torch.long
            ),

            "label": torch.tensor(
                label_id,
                dtype=torch.long
            )
        }

In [28]:
train_dataset = Banking77BiLSTMDataset(
    train_df,
    word_to_id,
    label_to_id,
    MAX_LENGTH
)

validation_dataset = Banking77BiLSTMDataset(
    validation_df,
    word_to_id,
    label_to_id,
    MAX_LENGTH
)

test_dataset = Banking77BiLSTMDataset(
    test_df,
    word_to_id,
    label_to_id,
    MAX_LENGTH
)

In [29]:
sample = train_dataset[0]

print(
    "Input shape:",
    sample["input_ids"].shape
)

print(
    "Real length:",
    sample["length"]
)

print(
    "Label:",
    sample["label"]
)

Input shape: torch.Size([50])
Real length: tensor(7)
Label: tensor(73)


In [30]:
BATCH_SIZE = 32


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [31]:
batch = next(
    iter(train_loader)
)

print(
    "input_ids:",
    batch["input_ids"].shape
)

print(
    "length:",
    batch["length"].shape
)

print(
    "labels:",
    batch["label"].shape
)

input_ids: torch.Size([32, 50])
length: torch.Size([32])
labels: torch.Size([32])


## Bli Lstm

In [32]:
EMBEDDING_DIM = 128
HIDDEN_DIM = 128
NUM_LAYERS = 1
DROPOUT_RATE = 0.5

NUM_CLASSES = 77

In [33]:
# Cell 12：定义BiLSTM模型

from torch.nn.utils.rnn import (
    pack_padded_sequence
)


class BiLSTMClassifier(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim,
        hidden_dim,
        num_classes,
        padding_idx,
        dropout_rate=0.5
    ):
        super().__init__()

        # 1. Embedding
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=padding_idx
        )

        # 2. BiLSTM
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        # 3. Dropout
        self.dropout = nn.Dropout(
            dropout_rate
        )

        # 4. Classification layer
        self.classifier = nn.Linear(
            hidden_dim * 2,
            num_classes
        )


    def forward(
        self,
        input_ids,
        lengths
    ):

        # --------------------------------
        # [batch, seq_len]
        # →
        # [batch, seq_len, embedding_dim]
        # --------------------------------

        embedded = self.embedding(
            input_ids
        )

        # --------------------------------
        # 忽略Padding部分
        # --------------------------------

        packed = pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        # --------------------------------
        # BiLSTM
        # --------------------------------

        packed_output, (
            hidden,
            cell
        ) = self.lstm(
            packed
        )

        # hidden shape:
        #
        # [2, batch, hidden_dim]
        #
        # hidden[0] = forward
        # hidden[1] = backward

        forward_hidden = hidden[0]

        backward_hidden = hidden[1]

        # --------------------------------
        # 拼接两个方向
        # --------------------------------

        sentence_representation = (
            torch.cat(
                [
                    forward_hidden,
                    backward_hidden
                ],
                dim=1
            )
        )

        # [batch, hidden_dim * 2]

        sentence_representation = (
            self.dropout(
                sentence_representation
            )
        )

        logits = self.classifier(
            sentence_representation
        )

        return logits

In [34]:
# Cell 13：创建BiLSTM

bilstm_model = BiLSTMClassifier(
    vocab_size=VOCAB_SIZE,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    num_classes=NUM_CLASSES,
    padding_idx=PAD_ID,
    dropout_rate=DROPOUT_RATE
)

bilstm_model = (
    bilstm_model.to(device)
)

print(bilstm_model)

BiLSTMClassifier(
  (embedding): Embedding(1459, 128, padding_idx=0)
  (lstm): LSTM(128, 128, batch_first=True, bidirectional=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (classifier): Linear(in_features=256, out_features=77, bias=True)
)


In [35]:
# Cell 14：测试前向传播

batch = next(
    iter(train_loader)
)

input_ids = (
    batch["input_ids"]
    .to(device)
)

lengths = (
    batch["length"]
    .to(device)
)

labels = (
    batch["label"]
    .to(device)
)

print(
    "Input IDs:",
    input_ids.shape
)

print(
    "Lengths:",
    lengths.shape
)

with torch.no_grad():

    logits = bilstm_model(
        input_ids,
        lengths
    )

print(
    "Output:",
    logits.shape
)

Input IDs: torch.Size([32, 50])
Lengths: torch.Size([32])
Output: torch.Size([32, 77])


In [36]:
# Cell 15：定义损失函数和优化器

import torch.optim as optim


criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    bilstm_model.parameters(),
    lr=1e-3
)

print(criterion)
print(optimizer)

CrossEntropyLoss()
Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)


In [37]:
# Cell 16：BiLSTM单个epoch训练

def train_bilstm_one_epoch(
    model,
    data_loader,
    criterion,
    optimizer,
    device
):

    model.train()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for batch in data_loader:

        input_ids = (
            batch["input_ids"]
            .to(device)
        )

        lengths = (
            batch["length"]
            .to(device)
        )

        labels = (
            batch["label"]
            .to(device)
        )

        # 清空梯度
        optimizer.zero_grad()

        # 前向传播
        logits = model(
            input_ids,
            lengths
        )

        # 计算损失
        loss = criterion(
            logits,
            labels
        )

        # 反向传播
        loss.backward()

        # 更新参数
        optimizer.step()

        # 累计loss
        total_loss += (
            loss.item()
            * labels.size(0)
        )

        # 预测类别
        predictions = (
            logits.argmax(dim=1)
        )

        total_correct += (
            predictions
            == labels
        ).sum().item()

        total_samples += (
            labels.size(0)
        )

    average_loss = (
        total_loss
        / total_samples
    )

    accuracy = (
        total_correct
        / total_samples
    )

    return (
        average_loss,
        accuracy
    )

In [38]:
# Cell 17：BiLSTM验证函数

from sklearn.metrics import f1_score


def evaluate_bilstm(
    model,
    data_loader,
    criterion,
    device
):

    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    all_predictions = []
    all_labels = []

    with torch.no_grad():

        for batch in data_loader:

            input_ids = (
                batch["input_ids"]
                .to(device)
            )

            lengths = (
                batch["length"]
                .to(device)
            )

            labels = (
                batch["label"]
                .to(device)
            )

            logits = model(
                input_ids,
                lengths
            )

            loss = criterion(
                logits,
                labels
            )

            total_loss += (
                loss.item()
                * labels.size(0)
            )

            predictions = (
                logits.argmax(dim=1)
            )

            total_correct += (
                predictions
                == labels
            ).sum().item()

            total_samples += (
                labels.size(0)
            )

            all_predictions.extend(
                predictions
                .cpu()
                .numpy()
            )

            all_labels.extend(
                labels
                .cpu()
                .numpy()
            )

    average_loss = (
        total_loss
        / total_samples
    )

    accuracy = (
        total_correct
        / total_samples
    )

    macro_f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )

    return (
        average_loss,
        accuracy,
        macro_f1
    )

In [39]:
# Cell 18：带Early Stopping训练BiLSTM

MAX_EPOCHS = 25
PATIENCE = 4

best_validation_f1 = -1.0
epochs_without_improvement = 0

bilstm_training_history = []


for epoch in range(
    1,
    MAX_EPOCHS + 1
):

    train_loss, train_accuracy = (
        train_bilstm_one_epoch(
            model=bilstm_model,
            data_loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            device=device
        )
    )

    (
        validation_loss,
        validation_accuracy,
        validation_macro_f1
    ) = evaluate_bilstm(
        model=bilstm_model,
        data_loader=validation_loader,
        criterion=criterion,
        device=device
    )

    bilstm_training_history.append(
        {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_accuracy": train_accuracy,
            "validation_loss": validation_loss,
            "validation_accuracy": validation_accuracy,
            "validation_macro_f1": validation_macro_f1
        }
    )

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Val Loss: {validation_loss:.4f} | "
        f"Val Acc: {validation_accuracy:.4f} | "
        f"Val Macro-F1: {validation_macro_f1:.4f}"
    )

    if (
        validation_macro_f1
        > best_validation_f1
    ):

        best_validation_f1 = (
            validation_macro_f1
        )

        epochs_without_improvement = 0

        torch.save(
            bilstm_model.state_dict(),
            "best_bilstm.pt"
        )

        print(
            "  → Best model updated"
        )

    else:

        epochs_without_improvement += 1

        print(
            f"  → No improvement "
            f"({epochs_without_improvement}/{PATIENCE})"
        )

        if (
            epochs_without_improvement
            >= PATIENCE
        ):

            print(
                "Early stopping triggered."
            )

            break

Epoch 01 | Train Loss: 3.0440 | Train Acc: 0.3197 | Val Loss: 1.6307 | Val Acc: 0.6089 | Val Macro-F1: 0.5919
  → Best model updated
Epoch 02 | Train Loss: 1.2086 | Train Acc: 0.6948 | Val Loss: 0.9579 | Val Acc: 0.7542 | Val Macro-F1: 0.7479
  → Best model updated
Epoch 03 | Train Loss: 0.7269 | Train Acc: 0.8121 | Val Loss: 0.7220 | Val Acc: 0.8047 | Val Macro-F1: 0.8045
  → Best model updated
Epoch 04 | Train Loss: 0.5010 | Train Acc: 0.8697 | Val Loss: 0.6188 | Val Acc: 0.8327 | Val Macro-F1: 0.8316
  → Best model updated
Epoch 05 | Train Loss: 0.3614 | Train Acc: 0.9044 | Val Loss: 0.5610 | Val Acc: 0.8434 | Val Macro-F1: 0.8441
  → Best model updated
Epoch 06 | Train Loss: 0.2657 | Train Acc: 0.9322 | Val Loss: 0.5364 | Val Acc: 0.8516 | Val Macro-F1: 0.8507
  → Best model updated
Epoch 07 | Train Loss: 0.1982 | Train Acc: 0.9511 | Val Loss: 0.5079 | Val Acc: 0.8547 | Val Macro-F1: 0.8537
  → Best model updated
Epoch 08 | Train Loss: 0.1736 | Train Acc: 0.9557 | Val Loss: 0.5114 

In [40]:
# Cell 19：加载最佳BiLSTM

bilstm_model.load_state_dict(
    torch.load(
        "best_bilstm.pt",
        map_location=device
    )
)

bilstm_model = (
    bilstm_model.to(device)
)

print(
    "Best validation Macro-F1:",
    best_validation_f1
)

Best validation Macro-F1: 0.8672171537368506


In [41]:
# Cell 20：BiLSTM最终测试集评估

(
    bilstm_test_loss,
    bilstm_test_accuracy,
    bilstm_test_macro_f1
) = evaluate_bilstm(
    model=bilstm_model,
    data_loader=test_loader,
    criterion=criterion,
    device=device
)

print(
    "BiLSTM Test Loss:",
    f"{bilstm_test_loss:.4f}"
)

print(
    "BiLSTM Test Accuracy:",
    f"{bilstm_test_accuracy:.4f}"
)

print(
    "BiLSTM Test Macro-F1:",
    f"{bilstm_test_macro_f1:.4f}"
)

BiLSTM Test Loss: 0.5051
BiLSTM Test Accuracy: 0.8587
BiLSTM Test Macro-F1: 0.8614
